## Exercises on Proximal Policy Optimization

These paper-and-pencil exercises reinforce Chapter 11: the gap between parameter steps and policy changes, the KL divergence as a measure of policy change, the performance difference lemma, the importance-sampled surrogate objective and its first-order agreement with the true objective, the TRPO lower bound, PPO's clipped surrogate, the approximate-KL diagnostic, and the accounting of data reuse. Notation: old policy $\pi_{\theta_{\text{old}}}$ (abbreviated $\pi_{\text{old}}$), candidate policy $\pi_\theta$, ratio $r_t(\theta)=\pi_\theta(A_t\mid S_t)/\pi_{\theta_{\text{old}}}(A_t\mid S_t)$, advantage estimate $A_t$, clipping range $\epsilon$, trust-region radius $\delta$.

### Exercise 11.1 — Same parameter step, different policy change

A two-action softmax policy with one preference parameter has $\pi_\theta(a_1\mid s)=1/(1+e^{-\theta})$. Consider the parameter step $\Delta\theta=1$ starting from (a) $\theta=0$ and (b) $\theta=4$.

1. Compute $\pi(a_1\mid s)$ before and after the step in both cases.
2. Compute the KL divergence $D_{\text{KL}}(\pi_{\text{old}}\,\|\,\pi_{\text{new}})$ in both cases.
3. What does this say about choosing a learning rate in parameter space?

**Step 1 — Probabilities.**

- (a) $\theta=0\to1$: $\ \pi(a_1)=0.5\to 1/(1+e^{-1}) = 0.7311$ (change $+0.231$).
- (b) $\theta=4\to5$: $\ \pi(a_1)=1/(1+e^{-4})=0.9820\to 1/(1+e^{-5})=0.9933$ (change $+0.011$).

**Step 2 — KL divergence** $D_{\text{KL}} = p\ln\frac{p}{p'}+(1-p)\ln\frac{1-p}{1-p'}$:

- (a) $0.5\ln\frac{0.5}{0.7311}+0.5\ln\frac{0.5}{0.2689} = 0.5(-0.3799)+0.5(0.6201) = 0.120$ nats.
- (b) $0.9820\ln\frac{0.9820}{0.9933}+0.0180\ln\frac{0.0180}{0.0067} = 0.9820(-0.01144)+0.0180(0.9883) = -0.01123+0.01779 = 0.0066$ nats.

The same parameter step produces a policy change **eighteen times larger** (in KL) in case (a) than in case (b).

**Step 3 — Consequence.** A learning rate chosen so that updates are safe near $\theta=0$ (where the policy is sensitive) wastes most steps near $\theta=4$ (where it is insensitive), and vice versa; in a neural network every parameter sits somewhere different on this spectrum in every state. The size of the update should therefore be controlled in **policy space** (KL divergence), not in parameter space, which is what trust regions and PPO's clipping do.

**Key concept**

The map from parameters to policies is highly non-uniform: a fixed step in $\theta$ is a tiny policy change in some states and a huge one in others. The KL divergence measures the change that matters, independently of the parameterisation.

### Exercise 11.2 — The performance difference lemma on a small MDP

A two-step episodic task, $\gamma=1$: from $s_0$ actions $L$ and $R$ lead deterministically to $s_L$ and $s_R$ with reward $0$; from $s_L$ the only action gives reward $1$, from $s_R$ it gives reward $3$; then the episode ends. The current policy is $\pi(L\mid s_0)=0.75$. The candidate policy is $\pi'(L\mid s_0)=0.25$.

1. Compute $v_\pi(s_0)$, $q_\pi(s_0,\cdot)$ and the advantages $a_\pi(s_0,\cdot)$.
2. Compute $J(\pi')-J(\pi)$ directly.
3. Verify the performance difference lemma $J(\pi')-J(\pi)=\mathbb E_{\tau\sim\pi'}\big[\sum_t\gamma^t a_\pi(S_t,A_t)\big]$.

**Step 1 — Values of $\pi$.** $\ q_\pi(s_0,L)=1$, $q_\pi(s_0,R)=3$, $\ v_\pi(s_0)=0.75(1)+0.25(3)=1.5$, so $a_\pi(s_0,L)=-0.5$, $a_\pi(s_0,R)=+1.5$. In $s_L$ and $s_R$ there is a single action, so the advantage is $0$.

**Step 2 — Direct computation.** $\ J(\pi)=v_\pi(s_0)=1.5$; $\ J(\pi')=0.25(1)+0.75(3)=2.5$; $\ J(\pi')-J(\pi)=1.0.$

**Step 3 — Lemma.** Under $\pi'$ the first action is $L$ with probability $0.25$ (advantage $-0.5$) and $R$ with probability $0.75$ (advantage $+1.5$); later advantages are zero:

$\displaystyle \mathbb E_{\tau\sim\pi'}\Big[\sum_t a_\pi(S_t,A_t)\Big] = 0.25(-0.5)+0.75(1.5) = -0.125+1.125 = 1.0.\ \checkmark$

Note the asymmetry: the advantages are those of the **old** policy, the actions are those of the **new** one.

**Key concept**

$J(\pi')-J(\pi)$ equals the expected discounted sum of the old policy's advantages along the new policy's trajectories. To improve, the new policy must prefer actions that the old policy's critic rates above average — exactly the information a rollout of $\pi$ gives us, except for *which states* $\pi'$ will visit.

### Exercise 11.3 — The surrogate objective and its gradient

Use the task of Exercise 11.2 with a softmax policy $\pi_\theta(L\mid s_0)=1/(1+e^{-\theta})$; the old policy has $\theta_{\text{old}}=\ln 3$ (so $\pi_{\text{old}}(L)=0.75$). Since the task is episodic with $\gamma=1$, the surrogate is written as a sum over the time steps rather than with the normalised state distribution and the $1/(1-\gamma)$ factor.

1. Write the surrogate $L_{\pi_{\text{old}}}(\pi_\theta)=\mathbb E_{a\sim\pi_{\text{old}}}\big[\tfrac{\pi_\theta(a\mid s_0)}{\pi_{\text{old}}(a\mid s_0)}\,a_\pi(s_0,a)\big]$ as a function of $p=\pi_\theta(L\mid s_0)$ and evaluate it at $p=0.75$ and $p=0.25$.
2. Compare the surrogate at $p=0.25$ with the true improvement of Exercise 11.2, and explain the agreement.
3. Show that $\mathrm d L/\mathrm d\theta$ at $\theta_{\text{old}}$ equals the policy gradient $\mathrm dJ/\mathrm d\theta$ at $\theta_{\text{old}}$.

**Step 1 — Surrogate.** Importance weights cancel the old probabilities: $\ L = 0.75\cdot\frac{p}{0.75}(-0.5) + 0.25\cdot\frac{1-p}{0.25}(1.5) = -0.5p+1.5(1-p) = 1.5-2p.$

At $p=0.75$: $L=0$ (the old policy improves on itself by nothing). At $p=0.25$: $L=1.0$.

**Step 2 — Agreement.** The surrogate predicts an improvement of $1.0$, which is **exactly** the true improvement. This is because the state-distribution approximation $d^{\pi'}\approx d^{\pi}$ is *exact* here: $s_0$ is always the first state, and the advantages in the later states are zero, so the states whose distribution changes do not contribute. In general the surrogate is only accurate for policies close to the old one.

**Step 3 — Gradient match.** $J(\theta) = p(1)+(1-p)(3) = 3-2p$ and $L(\theta)=1.5-2p$ differ by a constant, so $\mathrm dL/\mathrm d\theta = \mathrm dJ/\mathrm d\theta = -2\,\mathrm dp/\mathrm d\theta = -2p(1-p)$; at $\theta_{\text{old}}$: $-2(0.75)(0.25) = -0.375$ for both. (In general the two functions are not equal, but their gradients coincide at $\theta_{\text{old}}$, because the gradient of the ratio at $\theta_{\text{old}}$ is the score function.)

**Key concept**

The surrogate $\mathbb E_{\pi_{\text{old}}}[r(\theta)\,A]$ can be computed from the old policy's data for *any* $\theta$, matches the true objective to first order at $\theta_{\text{old}}$, and is exact only insofar as the new policy visits the same states. It is the quantity that both TRPO and PPO optimise.

### Exercise 11.4 — The TRPO lower bound

For two policies the bound $J(\pi')\ \ge\ J(\pi)+L_\pi(\pi')-C\,D_{\text{KL}}^{\max}(\pi\|\pi')$ holds with $C=\dfrac{4\gamma\,\epsilon_A}{(1-\gamma)^2}$, where $\epsilon_A=\max_{s,a}|a_\pi(s,a)|$. Take $\gamma=0.99$, $\epsilon_A=1$.

1. Compute $C$.
2. An update is predicted by the surrogate to gain $L_\pi(\pi')=2$, with $D_{\text{KL}}^{\max}=10^{-4}$. Does the bound guarantee improvement? What if $D_{\text{KL}}^{\max}=10^{-3}$?
3. Why does TRPO use a KL *constraint* with a radius $\delta\approx0.01$ rather than this penalty?

**Step 1 — Constant.** $\ C = \dfrac{4(0.99)(1)}{(0.01)^2} = \dfrac{3.96}{10^{-4}} = 39600.$

**Step 2 — Guarantees.** $\ D_{\text{KL}}^{\max}=10^{-4}$: penalty $=3.96$; bound $J(\pi')-J(\pi)\ge 2-3.96=-1.96$: **no guarantee**. $\ D_{\text{KL}}^{\max}=10^{-3}$: penalty $=39.6$, bound $\ge-37.6$: no guarantee either. Only for $D_{\text{KL}}^{\max}<2/39600\approx5\times10^{-5}$ would the bound certify improvement.

**Step 3 — Why a constraint.** With $C\sim 1/(1-\gamma)^2$ the penalty is enormous: following the bound literally would force minuscule steps and painfully slow learning. The bound is a *worst-case* statement (it uses the maximum KL over all states and the maximum advantage). TRPO keeps its structure — control the KL — but replaces the penalty by a hard constraint on the *average* KL with an empirically chosen radius ($\delta\approx0.01$), which gives much larger steps that are safe in practice. PPO goes one step further and drops the explicit KL altogether in favour of clipping.

**Key concept**

The theory says: improvement is guaranteed if the surrogate gain exceeds a KL penalty with a very large coefficient. The practical algorithms keep the *form* of this result (bound the KL) but not its constants: a trust region of radius $\delta$ (TRPO) or a clipping range $\epsilon$ (PPO).

### Exercise 11.5 — The clipped surrogate, case by case

With $\epsilon=0.2$, evaluate PPO's per-transition objective $\ell(r)=\min\big(rA,\ \text{clip}(r,0.8,1.2)\,A\big)$ and its derivative $\mathrm d\ell/\mathrm dr$ for the ratios $r\in\{0.5,\,0.9,\,1.0,\,1.1,\,1.5\}$, first with $A=+2$ and then with $A=-2$. Summarise in a table and explain where the gradient vanishes.

**$A=+2$.** The clipped term is $\text{clip}(r,0.8,1.2)\cdot 2$; its minimum with $2r$ is $2r$ for $r\le1.2$ and $2.4$ beyond.

| $r$ | $rA$ | clipped term | $\ell(r)$ | $\mathrm d\ell/\mathrm dr$ |
|---|---|---|---|---|
| $0.5$ | $1.0$ | $1.6$ | $1.0$ | $2$ |
| $0.9$ | $1.8$ | $1.8$ | $1.8$ | $2$ |
| $1.0$ | $2.0$ | $2.0$ | $2.0$ | $2$ |
| $1.1$ | $2.2$ | $2.2$ | $2.2$ | $2$ |
| $1.5$ | $3.0$ | $2.4$ | $2.4$ | $0$ |

A good action: the objective rewards increasing $r$ up to $1.2$, then **flattens**. Below $0.8$ (e.g. $r=0.5$) the unclipped term is the smaller one and the gradient is kept: a good action that has been made *less* likely can always be pushed back up.

**$A=-2$.** The clipped term is $\text{clip}(r,0.8,1.2)\cdot(-2)$; its minimum with $-2r$ is $-2r$ for $r\ge0.8$ and $-1.6$ below.

| $r$ | $rA$ | clipped term | $\ell(r)$ | $\mathrm d\ell/\mathrm dr$ |
|---|---|---|---|---|
| $0.5$ | $-1.0$ | $-1.6$ | $-1.6$ | $0$ |
| $0.9$ | $-1.8$ | $-1.8$ | $-1.8$ | $-2$ |
| $1.0$ | $-2.0$ | $-2.0$ | $-2.0$ | $-2$ |
| $1.1$ | $-2.2$ | $-2.2$ | $-2.2$ | $-2$ |
| $1.5$ | $-3.0$ | $-2.4$ | $-3.0$ | $-2$ |

A bad action: maximising $\ell$ means *decreasing* $r$ (gradient $-2$) until $r=0.8$, where the objective **flattens**; above $1.2$ (e.g. $r=1.5$) the unclipped term is the smaller one and the gradient is kept: a bad action that has been made *more* likely can always be pushed back down.

**Key concept**

The $\min$ makes $\ell$ a pessimistic bound of $rA$: the gradient vanishes exactly when the ratio has moved **in the advantage's direction** beyond $1\pm\epsilon$ (no incentive to go further), and is never switched off when the ratio has moved the **wrong** way (always possible to correct). This asymmetry is the whole mechanism of PPO.

### Exercise 11.6 — The approximate KL diagnostic

PPO implementations estimate the KL divergence between old and new policy from the ratios of a mini-batch as $\hat D = \mathbb E_t\big[(r_t-1)-\log r_t\big]$.

1. Show that $\mathbb E_{a\sim\pi_{\text{old}}}\big[(r-1)-\log r\big] = D_{\text{KL}}(\pi_{\text{old}}\,\|\,\pi_\theta)$ exactly, and that every term is non-negative.
2. A mini-batch has ratios $(0.9,\,1.1,\,1.3,\,0.7)$. Compute $\hat D$ and compare it with the naive estimator $-\mathbb E_t[\log r_t]$.
3. What value would make you reduce the number of epochs?

**Step 1 — Exactness and sign.** Since $r=\pi_\theta/\pi_{\text{old}}$, $\ \mathbb E_{a\sim\pi_{\text{old}}}[r] = \sum_a\pi_{\text{old}}\frac{\pi_\theta}{\pi_{\text{old}}} = \sum_a\pi_\theta = 1$, so $\mathbb E[r-1]=0$ and

$\displaystyle \mathbb E_{\pi_{\text{old}}}\big[(r-1)-\log r\big] = -\mathbb E_{\pi_{\text{old}}}\Big[\log\frac{\pi_\theta}{\pi_{\text{old}}}\Big] = \mathbb E_{\pi_{\text{old}}}\Big[\log\frac{\pi_{\text{old}}}{\pi_\theta}\Big] = D_{\text{KL}}(\pi_{\text{old}}\|\pi_\theta).$

Each term $(r-1)-\log r\ge0$ because $\log r\le r-1$ for all $r>0$ (the tangent line of $\log$ at $1$ lies above it), with equality only at $r=1$.

**Step 2 — Numbers.**

| $r$ | $r-1-\log r$ | $-\log r$ |
|---|---|---|
| $0.9$ | $-0.1+0.1054=0.0054$ | $0.1054$ |
| $1.1$ | $0.1-0.0953=0.0047$ | $-0.0953$ |
| $1.3$ | $0.3-0.2624=0.0376$ | $-0.2624$ |
| $0.7$ | $-0.3+0.3567=0.0567$ | $0.3567$ |

$\hat D = (0.0054+0.0047+0.0376+0.0567)/4 = 0.0261$, while $-\mathbb E[\log r] = (0.1054-0.0953-0.2624+0.3567)/4 = 0.0261$. The two coincide here only because these four ratios happen to sum exactly to $4$, so that $\sum(r-1)=0$ on this mini-batch; in general they differ sample by sample. The two estimators have the same expectation (the $r-1$ term has mean zero), but the naive one has terms of both signs and a much larger variance; the corrected one is non-negative term by term and far more stable on small mini-batches.

**Step 3 — Threshold.** Typical PPO runs keep $\hat D$ around $10^{-3}$ to $10^{-2}$ per update. A value above roughly $0.02$–$0.03$ (with $\epsilon=0.2$) indicates that the epochs are moving the policy further than the clipping was meant to allow: reduce the epochs or the learning rate, or stop the epochs early when the threshold is crossed.

**Key concept**

$(r-1)-\log r$ is an unbiased, non-negative, low-variance estimator of the KL divergence computable from the ratios PPO already has. Since clipping is a heuristic and not a constraint, this diagnostic is how one checks that the policy is *actually* staying close.

### Exercise 11.7 — Data reuse: A2C vs. PPO accounting

Chapter 09's A2C used $K=8$ environments and $n=20$ steps per rollout with a single gradient step; Chapter 11's PPO uses $K=8$, $n=128$, $E=4$ epochs and mini-batches of $256$.

1. For a budget of $150{,}000$ environment steps, compute for each algorithm the number of rollouts, the number of gradient steps, and the number of gradient steps per transition.
2. Each PPO transition is seen by how many gradient steps? Why is this harmless in PPO and not in A2C?
3. PPO's rollouts are eight times longer than A2C's. What is gained and what is lost?

**Step 1 — Counts.**

| | A2C | PPO |
|---|---|---|
| transitions per rollout | $8\times20=160$ | $8\times128=1024$ |
| rollouts in $150{,}000$ steps | $938$ (the last one completes the budget) | $147$ |
| gradient steps per rollout | $1$ | $4\times(1024/256)=16$ |
| total gradient steps | $938$ | $2352$ |
| gradient steps per transition | $1/160=0.006$ | $16/1024=0.016$ |

**Step 2 — Reuse.** Every PPO transition appears in exactly one mini-batch per epoch, so it contributes to $E=4$ gradient steps, computed with a policy that has already moved (ratios $\ne1$). In A2C the policy-gradient estimate is valid only for the policy that generated the data, so reusing a transition after an update would bias the gradient; PPO's importance ratios make the estimate valid off the old policy *as long as it stays close*, and the clipping keeps it close. The reuse is "free" exactly to the extent that the ratios remain near $1$.

**Step 3 — Longer rollouts.** Gained: a larger and more diverse batch (more episodes, more states) for each sequence of updates, less biased advantage estimates (the GAE sum is truncated later, so the bootstrap on the critic weighs less), and fewer rollout–update cycles. Lost: the policy is updated less often per environment step ($147$ times instead of $938$), so early in training, when each update brings a large gain, PPO can lag for the first few rollouts.

**Key concept**

PPO's sample efficiency over A2C comes from taking $E\times$(mini-batches) gradient steps on each rollout instead of one. The importance ratios make the extra steps legitimate, and the clipped objective keeps them from moving the policy out of the region where the ratios can be trusted.

### Exercise 11.8 — Ratios for a Gaussian policy (continuous actions)

A one-dimensional Gaussian policy with state-independent standard deviation $\sigma=0.5$ took the action $a=0.6$ when its mean was $\mu_{\text{old}}=0.2$. After some epochs the mean in that state is $\mu=0.5$ ($\sigma$ unchanged).

1. Compute the ratio $r=\pi_\theta(a)/\pi_{\text{old}}(a)$.
2. With $\epsilon=0.2$ and advantage $A=+1$, is this transition still contributing a gradient to the actor's mean?
3. The log standard deviation is a learned parameter. What happens to the ratios of *all* transitions if $\sigma$ is reduced from $0.5$ to $0.4$ while the means stay fixed, for an action at the mean and for one at $1.5\sigma$ from it?

**Step 1 — Ratio.** With equal $\sigma$, $\ \log r = -\dfrac{(a-\mu)^2-(a-\mu_{\text{old}})^2}{2\sigma^2} = -\dfrac{0.01-0.16}{0.5} = 0.30$, so $r=e^{0.30}=1.350$.

**Step 2 — Clipping.** $A>0$ and $r=1.35>1+\epsilon=1.2$: the clipped term $1.2A$ is the minimum, its gradient with respect to $\mu$ is zero. This transition no longer pushes the mean; the mean has already moved "enough" towards the good action $0.6$ (from $0.2$ to $0.5$) as far as this sample is concerned. Transitions whose ratio is still inside $[0.8,1.2]$ continue to drive the update.

**Step 3 — Changing $\sigma$.** The ratio of two Gaussians with the same mean and $\sigma_{\text{old}}=0.5$, $\sigma=0.4$ is $r = \dfrac{\sigma_{\text{old}}}{\sigma}\exp\Big(-\dfrac{(a-\mu)^2}{2}\big(\tfrac{1}{\sigma^2}-\tfrac{1}{\sigma_{\text{old}}^2}\big)\Big)$.

- Action at the mean ($a=\mu$): $r=0.5/0.4=1.25$ — made *more* likely (above $1+\epsilon$, so clipped if $A>0$).
- Action at $a-\mu=1.5\sigma_{\text{old}}=0.75$: exponent $-\tfrac{0.5625}{2}(6.25-4) = -0.633$, $r = 1.25\,e^{-0.633} = 0.664$ — made *less* likely (below $1-\epsilon$, so clipped if $A<0$).

Narrowing the distribution raises the ratio of central actions and lowers that of peripheral ones, so the clipping also limits how fast $\sigma$ can shrink: the standard deviation cannot collapse in a single rollout, which slows its decrease and lets PPO with a Gaussian policy keep exploring for a long time even without an entropy bonus.

**Key concept**

For continuous actions the ratio is a ratio of densities; moves of the mean and of the standard deviation both change it, and the clipping bounds both. In particular it prevents the learned $\sigma$ from collapsing abruptly, giving PPO a built-in, gradually decreasing exploration (though not a guarantee against collapse over many updates).